# Gold Layer DDL - Initial Table Creation

This notebook creates the initial schema for all fact and dimension tables in the `neo_bank.gold` layer.

**Purpose**: These tables must exist before running the incremental load and transformation notebooks, as the delta merge operations require pre-existing target tables.

## Tables Created:

**Dimension Tables:**
* `dim_branches` - Branch location and regional information
* `dim_customers` - Customer profile and KYC details
* `dim_accounts` - Account-level information with customer and branch relationships

**Fact Tables:**
* `fact_transactions` - Transaction activity (partitioned by `txn_date_key`)
* `fact_credit_bureau_reports` - Credit bureau data (partitioned by `bureau_pull_date_key`)
* `fact_payment_gateway_logs` - Payment gateway processing logs (partitioned by `processed_date_key`)

**Note**: All dimension tables use `GENERATED ALWAYS AS IDENTITY` for surrogate keys to ensure unique identifiers across incremental loads.

In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.gold.dim_branches (
    branch_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    branch_code STRING,
    branch_name STRING,
    city STRING,
    state STRING,
    region STRING
) USING DELTA

In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.gold.dim_customers (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    date_of_birth DATE,
    pan_number STRING,
    email STRING,
    phone_number STRING,
    kyc_status STRING,
    home_branch_sk BIGINT
) USING DELTA

In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.gold.dim_accounts (
    account_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    account_id BIGINT,
    customer_sk BIGINT,
    account_type STRING,
    balance DOUBLE,
    currency STRING,
    account_branch_sk BIGINT,
    status STRING,
    account_opened_date DATE
) USING DELTA

In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.gold.fact_transactions (
    txn_id BIGINT,
    txn_type STRING,
    channel STRING,
    status STRING,
    account_sk BIGINT,
    customer_sk BIGINT,
    txn_date_key INT,
    amount DOUBLE
) USING DELTA
PARTITIONED BY (txn_date_key)

In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.gold.fact_credit_bureau_reports (
    customer_sk BIGINT,
    credit_score INT,
    risk_grade STRING,
    external_active_loans INT,
    external_overdue_amount INT,
    bureau_pull_date_key INT
) USING DELTA
PARTITIONED BY (bureau_pull_date_key)

In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.gold.fact_payment_gateway_logs (
    txn_id STRING,
    gateway_name STRING,
    gateway_status STRING,
    response_code INT,
    device_type STRING,
    geo_location STRING,
    account_sk BIGINT,
    customer_sk BIGINT,
    processed_date_key INT,
    processing_time_ms INT
) USING DELTA
PARTITIONED BY (processed_date_key)